# BindsNET

BindsNET is a Python library for simulating spiking neural networks, built on top of PyTorch.

## Supported primitives

### Write to NIR (BindsNET → NIR)

| BindsNET | NIR |
| --- | --- |
| `Input` | `Input` |
| `LIFNodes` | `LIF` |
| `IFNodes` | `IF` |
| `Connection` (no bias) | `Linear` |
| `Connection` (with bias) | `Affine` |

### Read from NIR (NIR → BindsNET)

| NIR | BindsNET |
| --- | --- |
| `Input` | `Input` |
| `Output` | *(virtual, no layer)* |
| `Linear` | `Connection` |
| `Affine` | `Connection` (with bias) |
| `LIF` | `LIFNodes` |
| `IF` | `IFNodes` |

## Current limitations

- Only simple feedforward (acyclic, single-input single-output) graphs are supported
- Recurrent connections, branching, and merging are not yet supported
- Conv1d/Conv2d are not yet supported
- CubaLIF is not yet supported (no direct BindsNET equivalent)
- The resistance parameter `R` in NIR LIF/IF neurons is assumed to be 1.0 (folded into weights)
- BindsNET neuron parameters are treated as uniform across all neurons in a layer

## Import a NIR graph into BindsNET

In [ ]:
import numpy as np

import nir
from nir_bindsnet import from_nir

# Create a NIR network: Input -> Affine -> LIF -> Output
affine_weights = np.array([[1.0, 2.0], [3.0, 4.0]])
affine_bias = np.array([0.1, -0.1])
lif_tau = np.array([10.0, 10.0])
lif_r = np.array([1.0, 1.0])
lif_v_leak = np.array([0.0, 0.0])
lif_v_threshold = np.array([1.0, 1.0])

nir_network = nir.NIRGraph.from_list(
    nir.Affine(affine_weights, affine_bias),
    nir.LIF(lif_tau, lif_r, lif_v_leak, lif_v_threshold),
)

# Convert to BindsNET
bindsnet_network = from_nir(nir_network, dt=1.0)

print("Layers:", list(bindsnet_network.layers.keys()))
print("Connections:", list(bindsnet_network.connections.keys()))

## Export a BindsNET model to NIR

In [ ]:
import torch

from bindsnet.network import Network
from bindsnet.network.nodes import Input, LIFNodes
from bindsnet.network.topology import Connection
from nir_bindsnet import to_nir

# Create a BindsNET network
net = Network(dt=1.0, learning=False)

input_layer = Input(n=4)
lif_layer = LIFNodes(n=3, thresh=1.0, rest=0.0, reset=0.0, tc_decay=10.0)

net.add_layer(input_layer, name="input")
net.add_layer(lif_layer, name="lif")

w = torch.randn(4, 3)
conn = Connection(source=input_layer, target=lif_layer, w=w)
net.add_connection(conn, source="input", target="lif")

# Convert to NIR
nir_graph = to_nir(net)

print("NIR nodes:", list(nir_graph.nodes.keys()))
print("NIR edges:", nir_graph.edges)

# Save to file
nir.write("bindsnet_model.nir", nir_graph)

## Round-trip conversion

You can verify that parameters are preserved through a round-trip conversion:

In [ ]:
import numpy as np

import nir
from nir_bindsnet import from_nir, to_nir

# Create a NIR graph
weight = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
original = nir.NIRGraph(
    nodes={
        "input": nir.Input(input_type=np.array([3])),
        "linear": nir.Linear(weight=weight),
        "lif": nir.LIF(
            tau=np.ones(2) * 10.0,
            r=np.ones(2),
            v_leak=np.zeros(2),
            v_threshold=np.ones(2),
        ),
        "output": nir.Output(output_type=np.array([2])),
    },
    edges=[
        ("input", "linear"),
        ("linear", "lif"),
        ("lif", "output"),
    ],
)

# NIR -> BindsNET -> NIR
bn_net = from_nir(original)
result = to_nir(bn_net)

# Check LIF parameters preserved
lif_nodes = [n for n in result.nodes.values() if isinstance(n, nir.LIF)]
print("LIF tau:", lif_nodes[0].tau)  # Should be [10., 10.]
print("LIF v_threshold:", lif_nodes[0].v_threshold)  # Should be [1., 1.]

# Check weights preserved
linear_nodes = [
    n for n in result.nodes.values()
    if isinstance(n, (nir.Linear, nir.Affine))
]
print("Weights match:", np.allclose(linear_nodes[0].weight, weight))